# 4 · Benchmarking a Model on the Evaluation Subset

This notebook evaluates any chat model on the 500-paper evaluation subset with the protocol of the paper:
the baseline prompt, temperature 0.1 / top-p 0.5, and macro-averaged metrics over the four ECNA classes.

For full runs use the scripts in `scripts/evaluation/`; this notebook shows the same steps interactively.

**Requirements:** a GPU and the paper texts (downloaded below).

In [ ]:
# Install the package (run once). In Colab, clone the repository first:
# !git clone https://github.com/m2l-lab/Novelty-Reviewer && cd Novelty-Reviewer
# !pip install -q -e "Novelty-Reviewer[eval]"

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

from novelty_reviewer import ChatModel, download_openreview_pdf, eval_subset, load_novelty_dataset, parse_novelty_score
from novelty_reviewer.evaluation import NLIScorer, classification_metrics
from novelty_reviewer.model import DEFAULT_GENERATION
from novelty_reviewer.pdf import pdf_to_markdown
from novelty_reviewer.prompts import BASELINE_PROMPT

N = 50   # use 500 for the full evaluation subset
papers = eval_subset(load_novelty_dataset("test")).select(range(N)).to_pandas()

## Paper texts

Equivalent to `python scripts/evaluation/prepare_paper_texts.py`.

In [ ]:
texts = {}
for paper_id in tqdm(papers.paper_id):
    try:
        texts[paper_id] = pdf_to_markdown(download_openreview_pdf(paper_id, "papers"))
    except RuntimeError as e:
        print(e)
papers = papers[papers.paper_id.isin(texts)]

## Generating predictions

Replace `MODEL` with any Hugging Face chat model or a local checkpoint.

In [ ]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model = ChatModel(MODEL)

outputs = [model.generate([{"role": "user", "content": BASELINE_PROMPT.format(paper_text=texts[p])}],
                          **DEFAULT_GENERATION) for p in tqdm(papers.paper_id)]
papers["generated_text"] = outputs
papers["predicted_score"] = [parse_novelty_score(o) for o in outputs]

## Score prediction metrics

In [ ]:
metrics = classification_metrics(papers.novelty_score.tolist(), papers.predicted_score.tolist())
pd.Series(metrics).round(4)

For reference, the results reported in the paper on all 500 papers:

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| GPT-OSS-20B | 0.53 | 0.205 | 0.295 | 0.227 |
| Llama-3.1-8B-Instruct | 0.15 | 0.162 | 0.280 | 0.089 |
| **Novelty Reviewer** | **0.62** | **0.378** | **0.319** | **0.323** |

## Free-text quality

The entailment–contradiction score compares each generated assessment with the reference ECNA assessment.

In [ ]:
nli = NLIScorer()
print("E-C NLI score:", round(nli(papers.generated_text.tolist(), papers.novelty_summary.tolist()), 4))

## Predicted score distribution

Most general-purpose LLMs overestimate novelty compared with reviewers:

In [ ]:
pd.DataFrame({"model": papers.predicted_score.value_counts(normalize=True),
              "reviewers (ECNA)": papers.novelty_score.value_counts(normalize=True)}).sort_index().mul(100).round(1)